#### Distances between class maps (Steps, Hook, Floor), sessions pooled per subject

Take one frequency band and one trial. Average the band power of every channel over the window [-0.8, 0] s before hand contact: the trial becomes one point in a space with one axis per channel (128 axes). The trials of a class form a cloud of points, and the centre of that cloud (its centroid) is the class map. There are three classes: first step of each walk (Steps), Grasp Hook, Grasp Floor. All trials use the contralateral hand, and the sessions of a subject are pooled (subjects are never pooled, their arrays differ).

For each pair of classes, the distance between the two centroids is the Mahalanobis distance: the distance weighted by how trials of the same class scatter around their centroid. A shift along a direction where trials already scatter a lot counts little; a shift along a direction where they barely scatter counts a lot. With 128 correlated channels, the high-scatter directions are mostly "many channels go up together". The distance is cross-validated (each class is split into two independent halves of trials), so a class with few trials does not look farther away just because its centroid is noisier.

Predictions on the distances between centroids: `H_goal` if Floor is closer to Hook than to Steps; `H_kin` if Floor is closer to Steps than to Hook; non-selective if no pair is separated. A class with fewer than 5 pooled trials is not testable and its statistics are NaN. Inference is by label permutation within session, Holm-corrected across the five bands. Beta and gamma are the primary bands; delta, theta and alpha are not separable at the 2 Hz resolution of the 0.5 s window.

#### Distances and permutation tests for every subject

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from src.config import SUBJECTS
from src.similarity import compute_class_distances

class_distance_results = {}
for subject in SUBJECTS:
    class_distance_results[subject] = compute_class_distances(subject)

summary_rows = []
for subject, subject_results in class_distance_results.items():
    for band, band_results in subject_results.items():
        totals = band_results['counts'].sum(axis=1)
        summary_rows.append({
            'subject': subject, 'band': band,
            'n_hook': totals['hook'], 'n_floor': totals['floor'], 'n_first_step': totals['first_step'],
            'lambda': band_results['lambda'],
            **{f'{name}': band_results['observed'][name] for name in band_results['observed']},
            **{f'p_holm_{name}': band_results['p_holm'][name] for name in band_results['p_holm']},
        })
pd.DataFrame(summary_rows).round(4)

[Router] Trial selection (contralateral hand, first step of each walk):
  Router_20211130: 21 walks, 21 first steps kept by the multitaper extraction, 15 after the hand filter.
  Router_20220211: 26 walks, 25 first steps kept by the multitaper extraction, 17 after the hand filter.
            Router_20211130  Router_20220211
hook                      2                5
floor                     3                8
first_step               15               17
[Router] delta: observed distance and 1000 permutations


c:\Users\tommy\miniconda3\envs\MonkeyReachWalk\Lib\site-packages\numpy\_core\numeric.py:475: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


[Router] theta: observed distance and 1000 permutations
[Router] alpha: observed distance and 1000 permutations
[Router] beta: observed distance and 1000 permutations
[Router] gamma: observed distance and 1000 permutations
[Wifi] Trial selection (contralateral hand, first step of each walk):
  Wifi_20210618: 7 walks, 7 first steps kept by the multitaper extraction, 2 after the hand filter.
  Wifi_20221020: 13 walks, 13 first steps kept by the multitaper extraction, 6 after the hand filter.
            Wifi_20210618  Wifi_20221020
hook                    4              1
floor                  12             11
first_step              2              6
[Wifi] delta: observed distance and 1000 permutations


c:\Users\tommy\miniconda3\envs\MonkeyReachWalk\Lib\site-packages\numpy\_core\numeric.py:475: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


[Wifi] theta: observed distance and 1000 permutations
[Wifi] alpha: observed distance and 1000 permutations
[Wifi] beta: observed distance and 1000 permutations
[Wifi] gamma: observed distance and 1000 permutations


,subject,band,n_hook,n_floor,n_first_step,lambda,d2_SH,d2_SF,d2_HF,delta_SH_minus_SF,delta_SF_minus_HF,p_holm_d2_SH,p_holm_d2_SF,p_holm_d2_HF,p_holm_delta_SH_minus_SF,p_holm_delta_SF_minus_HF
0,Router,delta,7,11,32,0.3140,0.2827,0.4088,0.4054,-0.1261,0.0034,0.1199,0.005,0.0659,1.0000,1.0000
1,Router,theta,7,11,32,0.3104,0.2690,0.2940,0.3008,-0.0250,-0.0068,0.2118,0.006,0.4366,1.0000,1.0000
2,Router,alpha,7,11,32,0.4580,0.2263,0.1590,0.3329,0.0673,-0.1739,0.1199,0.044,0.0639,1.0000,0.1718
3,Router,beta,7,11,32,0.3887,0.1790,0.1869,0.2685,-0.0079,-0.0816,0.3626,0.044,0.3836,1.0000,1.0000
4,Router,gamma,7,11,32,0.3396,0.5460,0.3593,0.5903,0.1867,-0.2311,0.0050,0.005,0.0050,0.2897,0.1548
5,Wifi,delta,5,23,8,0.4544,0.3666,0.3432,0.4457,0.0234,-0.1026,0.6953,0.005,0.0120,1.0000,1.0000
6,Wifi,theta,5,23,8,0.7016,0.3249,0.2360,0.3282,0.0889,-0.0922,0.6893,0.005,0.0050,1.0000,1.0000
7,Wifi,alpha,5,23,8,0.6590,0.3945,0.2003,0.4071,0.1942,-0.2068,0.6274,0.023,0.0120,0.9990,0.6893
8,Wifi,beta,5,23,8,0.6544,0.4496,0.3377,0.2883,0.1119,0.0494,0.6893,0.005,0.0120,1.0000,1.0000
9,Wifi,gamma,5,23,8,0.2584,1.1022,0.5086,0.8712,0.5935,-0.3626,0.2697,0.005,0.0050,0.1998,1.0000


#### Distances and null distributions for one subject and band

In [2]:
%matplotlib qt
from src.plot import plot_class_distances

subject = "Router"
band = "gamma"

plot_class_distances(class_distance_results[subject], subject, band)

#### Difference between class-mean maps (descriptive)

Class-mean band power minus class-mean band power, per channel and time, in session-z units. Channels are ordered on a random half of the trials and drawn from the other half. If Floor is closer to Steps, the Steps - Floor panel is paler than Steps - Hook.

In [3]:
%matplotlib qt
from src.plot import plot_pairwise_difference_heatmaps

subject = "Router"
band = "gamma"

plot_pairwise_difference_heatmaps(subject, band)